In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data.head(3)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1


In [4]:
data = data.drop(columns = ['RowNumber','CustomerId','Surname'])

In [5]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [15]:
one_hot_encoder_geo = OneHotEncoder(sparse_output=False)
ohe_geo = one_hot_encoder_geo.fit_transform(data[['Geography']])

In [16]:
ohe_geo_df = pd.DataFrame(ohe_geo, columns= one_hot_encoder_geo.get_feature_names_out())

In [19]:
data = pd.concat([data.drop(['Geography'], axis =1), ohe_geo_df], axis=1)
data.head(3)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0


In [20]:
X = data.drop('EstimatedSalary', axis = 1)
y = data['EstimatedSalary']

In [21]:
X_train,X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [23]:
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('one_hot_encoder_geo.pkl', 'wb') as file:
    pickle.dump('one_hot_encoder_geo',file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump('scaler', file)

## ANN REGRESION 

In [24]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

In [33]:
model = Sequential([
    Dense(64, activation='relu', input_shape = (X_train.shape[1],)),
    Dense(32, activation = 'relu'),
    Dense(1)  ## By default uses linear activation function
])

##Compile the model
model.compile(optimizer = 'adam', loss = 'mean_absolute_error', metrics = ['mae'])

model.summary()

Model: "sequential_1"


_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [37]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
log_dir = 'regressionlogs/fit/'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir= log_dir, histogram_freq=1)

In [35]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience = 10, restore_best_weights=True)

In [38]:
history = model.fit(X_train, y_train, validation_data=(X_test,y_test),epochs=100,
                    callbacks = [early_stopping_callback,tensorboard_callback])

Epoch 1/100
250/250 [==============================] - 0s 2ms/step - loss: 51298.2695 - mae: 51298.2695 - val_loss: 50456.8672 - val_mae: 50456.8672
Epoch 2/100
250/250 [==============================] - 0s 1ms/step - loss: 51029.0625 - mae: 51029.0625 - val_loss: 50395.6133 - val_mae: 50395.6133
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 51172.4922 - mae: 51172.4922 - val_loss: 50924.2422 - val_mae: 50924.2422
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 51286.3555 - mae: 51286.3555 - val_loss: 50887.7227 - val_mae: 50887.7227
Epoch 5/100
250/250 [==============================] - 1s 2ms/step - loss: 51054.4102 - mae: 51054.4102 - val_loss: 50418.2109 - val_mae: 50418.2109
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 51126.4688 - mae: 51126.4688 - val_loss: 50318.2109 - val_mae: 50318.2109
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 51147.8633 - mae: 51147.8633 - 

In [39]:
%load_ext tensorboard

In [40]:
##Evaluate model on test data
test_loss , test_mae = model.evaluate(X_test,y_test)
print(f'Test MAE : {test_mae}')

63/63 [==============================] - 0s 1ms/step - loss: 50318.2109 - mae: 50318.2109
Test MAE : 50318.2109375


In [41]:
model.save('regression_model.h5')

d:\D Drive Data\Github Projects\churn_ann_classification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
